# A search on Kaggle: one study (MLP, or one GNN operator on one feature configuration)

Runs on a Kaggle GPU notebook. The search strategy is the same for every model (`conf/hyperparameters/search.yaml`): Optuna TPE with the Hyperband pruner, 100 trials. The study is kept in `study.log`, so a search that does not fit one 12-hour session continues in the next one.
- Settings (right panel): Accelerator = GPU, Internet = On.
- Input: the dataset `sandrokhizanishvili/hi-small-gnn`.
- Set the cell below, then Run All (or Save Version -> Save & Run All to run headless).
- To continue a search: add the Output of the previous session as an input (Add Input -> Notebook Output Files), set `PREVIOUS` to its `outputs` folder, and run again.
- Everything is written to `/kaggle/working/outputs/tuning/<model>/<run>/`: `study.log`, `trials.csv`, `tuning_curves.png`, and `recipe.json` when the last trial ends.

In [ ]:
MODEL = 'mlp'            # mlp | gin | pna | gat | transformer
FEATURE_CONFIG = 'c2'    # a GNN only: c2 | c4
N_TRIALS = 100           # the budget of the study; 10 for a pilot
PREVIOUS = ''            # the outputs folder of the previous session (under /kaggle/input/...), '' on the first one

In [ ]:
# The pinned torch / PyG stack known to work on Kaggle GPU; do not change versions.
!pip uninstall -y torch torchvision torchaudio torch-scatter torch-sparse pyg_lib
!pip install torch==2.8.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.8.0+cu126.html
!pip install torch_geometric lightgbm==4.6.0 hydra-core==1.3.2 optuna==5.0.0

In [ ]:
# The code: clone once, pull on a rerun. Then the study of the previous session, if there is one.
import os
if os.path.isdir('/kaggle/working/gnn_for_aml'):
    !cd /kaggle/working/gnn_for_aml && git pull
else:
    !git clone https://github.com/sandrokhizanishvili/gnn_for_aml.git /kaggle/working/gnn_for_aml
%cd /kaggle/working/gnn_for_aml
!ls /kaggle/input
if PREVIOUS:
    !mkdir -p /kaggle/working/outputs && cp -rn {PREVIOUS}/tuning /kaggle/working/outputs/ && ls /kaggle/working/outputs/tuning/*/*

In [ ]:
# Environment test: versions, the GPU, every operator with the fixed and with searched settings.
!python run.py mode=check machine=kaggle

In [ ]:
# The search. Trials already in study.log are kept; a GNN trial cut by the session limit continues from its last epoch.
tuning = 'mlp/search' if MODEL == 'mlp' else f'gnn/search model={MODEL} feature_config={FEATURE_CONFIG}'
!python run.py tuning={tuning} hyperparameters.search.n_trials={N_TRIALS} machine=kaggle

In [ ]:
# Every trial of the study so far, best first (a cut session has no trials.csv yet: the study file is read instead).
import glob, optuna, pandas as pd
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend, JournalFileOpenLock
path = glob.glob(f'/kaggle/working/outputs/tuning/{MODEL}/*/study.log')[0]
study = optuna.load_study(study_name=os.path.basename(os.path.dirname(path)),
                          storage=JournalStorage(JournalFileBackend(path, lock_obj=JournalFileOpenLock(path))))
trials = study.trials_dataframe()
print(trials['state'].value_counts().to_dict())
trials.sort_values('value', ascending=False).head(15)

In [ ]:
# The MLP only: the final refit of the winner on train + val, seeds 42-46. A finished seed is skipped.
if MODEL == 'mlp' and glob.glob('/kaggle/working/outputs/tuning/mlp/*/recipe.json'):
    !python run.py -m tuning=mlp/final machine=kaggle

In [ ]:
# A small archive for download: everything except the checkpoints and predictions (the Output tab has the full folder).
!cd /kaggle/working && rm -f results_small.zip && zip -qr results_small.zip outputs -x "*.pt" "*predictions.csv" && ls -lh results_small.zip

## After the run
- Output tab: download `results_small.zip` or the whole `outputs` folder.
- Unzip into `gnn_for_aml/outputs/` on the laptop; commit the small files (`predictions.csv` and `*.pt` are gitignored); mark the row in `EXPERIMENTS.md`.
- A search that is not finished (no `recipe.json`): start a new session with `PREVIOUS` set, as described at the top.